<a href="https://colab.research.google.com/github/mryab/efficient-dl-systems/blob/main/week05_large_models/practice_part2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Efficient DL Practice: Advanced Parallelism (5 points)

In this practice session, we'll cover techniques for training large models in parallel: **Model** and **Sequence Parallelism**.
More precisely, you will implement them, and we will root for you as you go. Good luck, 🥩👜!



In [1]:
# dependencies: the code will likely work with slightly newer/older versions, but may require minimal patching
%pip install -q transformers==4.48.3 peft==0.14.0

import transformers; assert transformers.__version__.startswith("4.48"), transformers.__version__
import peft; assert peft.__version__.startswith("0.14"), peft.__version__

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.8/374.8 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 56.2 MB/s eta 0:00:00


__Part 1: Tensor Parallelism (2 points)__
![img](https://pytorch.org/tutorials/_images/megatron_lm.png)

We'll begin by implementing a simple tensor parallelism (also known as the [original](https://papers.nips.cc/paper_files/paper/2012/hash/c399862d3b9d6b76c8436e924a68c45b-Abstract.html) model parallelism).

Our ultimate objective is to run and fine-tune a Llama 3.x model in tensor-parallel mode. However, it is rather difficult to do that in one go, especially if you take bugs into account. So we'll start simple: __here's a single Llama MLP module:__

`please read the code below carefully, it's a template for the remaining assgnments`.

In [2]:
%%writefile tensor_parallel_mlp.py
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributed as dist


class LlamaMLP(nn.Module):  #  based on llama 3.1 8B configuration
    def __init__(self, hidden_size: int = 4096, intermediate_size: int = 14336):
        super().__init__()
        self.gate_proj = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.up_proj = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.down_proj = nn.Linear(intermediate_size, hidden_size, bias=False)

    def forward(self, input):
        return self.down_proj(F.silu(self.gate_proj(input)) * self.up_proj(input))


class ComputeWithAllReduce(torch.autograd.Function):
    @staticmethod  # fun fact: torch.distributed.nn has differentiable all_reduce!
    def forward(ctx, tp_shard: nn.Module, input: torch.Tensor):
        input = input.detach().requires_grad_(input.requires_grad)
        ctx.save_for_backward(input)
        ctx._tp_shard = tp_shard
        output = tp_shard(input)
        dist.all_reduce(output)
        return output
    @staticmethod
    def backward(ctx, grad_output: torch.Tensor):
        with torch.enable_grad():
          output = ctx._tp_shard(ctx.saved_tensors[0])
          output.backward(grad_output)
        dist.all_reduce(ctx.saved_tensors[0].grad)
        return None, ctx.saved_tensors[0].grad


class AllReduceModule(nn.Sequential):
    def forward(self, input: torch.Tensor):
        return ComputeWithAllReduce.apply(super().forward, input)


if __name__ == "__main__":
    dist.init_process_group("nccl")   # use nccl for cuda devices
    torch.manual_seed(1337)           # init weights equally on all ranks
    rank, world_size = dist.get_rank(), dist.get_world_size()

    for active_rank in range(world_size):
      dist.barrier()  # initialize each rank sequentially to save system RAM
      if rank != active_rank: continue

      # we will now implement Tensor Parallelism for the ref_module below:
      ref_module = nn.Sequential(nn.RMSNorm(4096), LlamaMLP())
      # compute reference tensors to test against them later
      input = torch.randn(1, 4096, requires_grad=True)
      ref_output = ref_module(input)
      ref_output.sum().backward()
      ref_input_grad = input.grad.clone()

      # TP step 1: define a module that computes a portion of intermediate units
      intermediate_size = ref_module[1].down_proj.in_features
      local_units = intermediate_size // world_size
      assert intermediate_size % world_size == 0
      tp_module = nn.Sequential(   # assign a portion of units per rank --v
          nn.RMSNorm(4096), AllReduceModule(LlamaMLP(intermediate_size=local_units))
      )   # all-reduce outputs during forward, all-reduce gradients on backward

      with torch.no_grad():  # copy select weights from the reference MLP
        # v-- input norm layer is too small to bother parallelizing - we replicate it!
        tp_module[0].load_state_dict(ref_module[0].state_dict())
        # up and gate projections are sharded across output units
        unit_slice = slice(local_units * rank, local_units * (rank + 1))
        tp_module[1][0].up_proj.weight[...] = ref_module[1].up_proj.weight[unit_slice]
        tp_module[1][0].gate_proj.weight[...] = ref_module[1].gate_proj.weight[unit_slice]
        # down projection is sharded across input units, matching up/gate proj
        tp_module[1][0].down_proj.weight[...] = ref_module[1].down_proj.weight[:, unit_slice]
      print(f"Initialized {rank=}", flush=True)
      del ref_module  # free RAM for next rank

    dist.barrier()  # test 1: forward pass
    tp_input = input.detach().requires_grad_(True)
    tp_output = tp_module(tp_input)
    if rank == 0:
        print(f"\nReference outputs ({rank=}):", ref_output.data, flush=True)
    for i in range(world_size):
        dist.barrier()
        if i != rank: continue
        print(f"TParallel outputs ({rank=}):", tp_output.data, flush=True)
        assert torch.allclose(tp_output, ref_output, atol=1e-6), f"output mismatch on {rank=}"

    dist.barrier()  # test 2: backward w.r.t. inputs
    assert tp_input.grad is None
    tp_output.sum().backward()
    if rank == 0:
        print(f"\nReference input grad ({rank=}):", ref_input_grad, flush=True)
    for i in range(world_size):
        dist.barrier()
        if i != rank: continue
        print(f"TParallel input grad ({rank=}):", tp_input.grad.data, flush=True)
        assert torch.allclose(tp_input.grad, ref_input_grad, atol=1e-6), f"input_grad mismatch on {rank=}"


Writing tensor_parallel_mlp.py


In [3]:
!OMP_NUM_THREADS=1 torchrun --nproc_per_node 1 tensor_parallel_mlp.py

/usr/local/lib/python3.12/dist-packages/torch/distributed/distributed_c10d.py:4807: UserWarning: No device id is provided via `init_process_group` or `barrier `. Using the current device set by the user. 
  warnings.warn(  # warn only once
[rank0]:[W920 12:00:50.862178174 ProcessGroupNCCL.cpp:5023] [PG ID 0 PG GUID 0 Rank 0]  using GPU 0 as device used by this process is currently unknown. This can potentially cause a hang if this rank to GPU mapping is incorrect. You can specify device_id in init_process_group() to force use of a particular device.
Initialized rank=0
[rank0]: Traceback (most recent call last):
[rank0]:   File "/content/tensor_parallel_mlp.py", line 80, in <module>
[rank0]:     tp_output = tp_module(tp_input)
[rank0]:                 ^^^^^^^^^^^^^^^^^^^
[rank0]:   File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1773, in _wrapped_call_impl
[rank0]:     return self._call_impl(*args, **kwargs)
[rank0]:            ^^^^^^^^^^^^^^^^^^^^^^^^^^^

Note that the code above lacks two details:
- it uses a form of checkpointing, but does not save random state, which would be required if you use dropout;
- it replicates RMSNorm, but it is not synchronized. Training would require all-reduce-ing gradients for those layers, e.g. by wrapping them with DDP.

```

```

```

```

```

```


__Task 1 (1 point):__ Implement tensor-parallel multi-head attention.

Like with the MLP module before, you can partition attention across multiple devices. This time, every device is to compute a portion of whole attention **heads** (and not individual units). We exploit the property that an multi-head attention layer can be viewed as a sum of individual head outputs after output projection.

For the sake of formality, this is the computation you need to parallelize:

In [4]:
import torch
from transformers.models.llama.modeling_llama import LlamaConfig, LlamaAttention, LlamaRotaryEmbedding
MODEL_NAME = "unsloth/Llama-3.2-1B"  # for testing (but not grading!), you may want to use Maykeye/TinyLLama-v0
config = LlamaConfig.from_pretrained(MODEL_NAME)
layer = LlamaAttention(config, layer_idx=5)
rotary_emb = LlamaRotaryEmbedding(config)

input = torch.randn(1, 128, config.hidden_size, requires_grad=True)
position_embeddings = rotary_emb(input, position_ids=torch.arange(128)[None])

output, *_etc = layer(input, attention_mask=None, position_embeddings=position_embeddings)
print(f"{output=}")
output.norm().backward()
print(f"{input.grad=}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/889 [00:00<?, ?B/s]

output=tensor([[[ 0.0182, -0.0556,  0.0225,  ..., -0.0273, -0.0202, -0.0813],
         [ 0.0126, -0.0733,  0.0327,  ..., -0.0263, -0.0312, -0.0461],
         [ 0.0101, -0.0599,  0.0368,  ..., -0.0395, -0.0344, -0.0699],
         ...,
         [ 0.0132, -0.0561,  0.0380,  ..., -0.0358, -0.0462, -0.0717],
         [ 0.0068, -0.0600,  0.0175,  ..., -0.0036, -0.0171, -0.0536],
         [ 0.0151, -0.0718,  0.0436,  ..., -0.0086, -0.0506, -0.0615]]],
       grad_fn=<UnsafeViewBackward0>)
input.grad=tensor([[[-0.0005,  0.0017,  0.0028,  ..., -0.0022,  0.0002,  0.0009],
         [-0.0005,  0.0017,  0.0029,  ..., -0.0021,  0.0004,  0.0008],
         [-0.0006,  0.0017,  0.0029,  ..., -0.0021,  0.0004,  0.0008],
         ...,
         [-0.0004,  0.0017,  0.0029,  ..., -0.0021,  0.0003,  0.0009],
         [-0.0003,  0.0018,  0.0029,  ..., -0.0020,  0.0003,  0.0009],
         [-0.0004,  0.0017,  0.0029,  ..., -0.0022,  0.0002,  0.0008]]])


Same as before, your task is to create a multi-head attention layer, partition it across ranks and verify two things:
- attention outputs on the same inputs (and mask) match with the non-parallel version;
- gradients w.r.t. attention inputs are the same; gradients w.r.t. mask need not be verified.


In [ ]:
%%writefile tensor_parallel_attn.py
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributed as dist


class MyLlamaAttention(nn.Module):
    ...  # please take a reference implementation of Llama attention from Hugging Face transformers:
    # https://github.com/huggingface/transformers/blob/v4.44-release/src/transformers/models/llama/modeling_llama.py#L326-L455
    # You can also directly import transformers.models.llama.modeling_llama.LlamaAttention, as in the reference above.
    # Alternatively, you are welcome to simplify their code or implement your own version.

    # Note: the link above points to an older version of attention with built-in rotary position embeddings (RoPE);
    # If you are using a newer version, please make sure to define extra inputs


# You will likely need to define additional classes below, e.g. a module to perform all-reduce


if __name__ == "__main__":
    dist.init_process_group("gloo")   # use nccl for cuda devices
    torch.manual_seed(1337)           # init weights equally on all ranks
    rank, world_size = dist.get_rank(), dist.get_world_size()

    for active_rank in range(world_size):
      dist.barrier()  # initialize each rank sequentially to save system RAM
      if rank != active_rank: continue

      # we will now implement Tensor Parallelism for the ref_module below:
      ref_module = MyLlamaAttention()
      # ^-- you may need to modify this code, e.g. pass parameters or use transformers LlamaAttention (as above)

      # generate reference tensors to test against them later
      input = torch.randn(1, 128, 4096, requires_grad=True)
      extra_inputs = dict()  # <-- OPTIONAL: either design additional inputs here, as in the reference above

      ref_output = ref_module(input, **extra_inputs)
      ref_output.sum().backward()
      ref_input_grad = input.grad.clone()

      # TP step 1: define a module that computes a portion of attention heads

      tp_module = <YOUR CODE HERE>  # create a tensor-parallel version of the Attention module

      with torch.no_grad():
          <YOUR CODE HERE>  # copy select weights from the reference attention

      print(f"Initialized {rank=}", flush=True)
      del ref_module  # free RAM for next rank

    # TEST AREA: you are free to add additional parameters, but your code *must* run the same tests as below
    dist.barrier()  # test 1: forward pass
    tp_input = input.detach().requires_grad_(True)
    tp_output = tp_module(tp_input, **extra_inputs)
    if rank == 0:
        print(f"\nReference outputs ({rank=}):", ref_output.data, flush=True)
    for i in range(world_size):
        dist.barrier()
        if i != rank: continue
        print(f"TParallel outputs ({rank=}):", tp_output.data, flush=True)
        assert torch.allclose(tp_output, ref_output, atol=1e-5), f"output mismatch on {rank=}"

    dist.barrier()  # test 2: backward w.r.t. inputs
    assert tp_input.grad is None
    tp_output.sum().backward()
    if rank == 0:
        print(f"\nReference input grad ({rank=}):", ref_input_grad, flush=True)
    for i in range(world_size):
        dist.barrier()
        if i != rank: continue
        print(f"TParallel input grad ({rank=}):", tp_input.grad.data, flush=True)
        assert torch.allclose(tp_input.grad, ref_input_grad, atol=1e-4), f"input_grad mismatch on {rank=}"


In [ ]:
!OMP_NUM_THREADS=1 torchrun --nproc_per_node 2 tensor_parallel_attn.py
# ^-- feel free to modify parameters, as long as there are at least 2 ranks

In [15]:
%%writefile tensor_parallel_attn.py
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributed as dist


# ---------------- Reference (упрощённая LLaMA-подобная) ----------------
class MyLlamaAttention(nn.Module):
    """Простая реализация multi-head attention в стиле LLaMA.
       Размерности: (B, S, H) -> (B, S, H)."""
    def __init__(self, hidden_size: int = 4096, num_heads: int = 32):
        super().__init__()
        assert hidden_size % num_heads == 0
        self.hidden_size = hidden_size
        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads
        self.wq = nn.Linear(hidden_size, hidden_size, bias=False)
        self.wk = nn.Linear(hidden_size, hidden_size, bias=False)
        self.wv = nn.Linear(hidden_size, hidden_size, bias=False)
        self.wo = nn.Linear(hidden_size, hidden_size, bias=False)

    def _shape(self, x):  # (B,S,H) -> (B, nH, S, D)
        B, S, _ = x.shape
        return x.view(B, S, self.num_heads, self.head_dim).transpose(1, 2)

    def forward(self, x):
        q = self._shape(self.wq(x))
        k = self._shape(self.wk(x))
        v = self._shape(self.wv(x))
        y = F.scaled_dot_product_attention(q, k, v)     # (B, nH, S, D)
        B, nH, S, D = y.shape
        y = y.transpose(1, 2).reshape(B, S, nH * D)      # (B, S, H)
        return self.wo(y)


# ---------------- Tensor Parallel: shard per heads ----------------
class TPAttentionShard(nn.Module):
    """Один ранг считает свой subset голов и свой кусок o_proj."""
    def __init__(self, hidden_size: int, num_heads: int, rank: int, world_size: int):
        super().__init__()
        assert num_heads % world_size == 0
        assert hidden_size % num_heads == 0
        self.hidden_size = hidden_size
        self.num_heads_total = num_heads
        self.rank = rank
        self.world_size = world_size

        self.head_dim = hidden_size // num_heads
        self.local_heads = num_heads // world_size
        self.local_hidden = self.local_heads * self.head_dim

        # column-parallel q,k,v (выход = только наши головы)
        self.wq = nn.Linear(hidden_size, self.local_hidden, bias=False)
        self.wk = nn.Linear(hidden_size, self.local_hidden, bias=False)
        self.wv = nn.Linear(hidden_size, self.local_hidden, bias=False)
        # row-parallel o (вход = только наши головы)
        self.wo = nn.Linear(self.local_hidden, hidden_size, bias=False)

    def _shape_local(self, x):  # (B,S,local_hidden) -> (B, h_loc, S, D)
        B, S, _ = x.shape
        return x.view(B, S, self.local_heads, self.head_dim).permute(0, 2, 1, 3)

    def forward(self, x):
        q = self._shape_local(self.wq(x))
        k = self._shape_local(self.wk(x))
        v = self._shape_local(self.wv(x))
        y = F.scaled_dot_product_attention(q, k, v)      # (B, h_loc, S, D)
        B, h, S, D = y.shape
        y = y.permute(0, 2, 1, 3).reshape(B, S, h * D)    # (B, S, local_hidden)
        return self.wo(y)                                 # (B, S, H) частичный вклад


class ComputeTPAttention(torch.autograd.Function):
    """Суммируем частичные выходы; в backward суммируем ∂L/∂x."""
    @staticmethod
    def forward(ctx, shard: TPAttentionShard, x: torch.Tensor):
        x = x.detach().requires_grad_(x.requires_grad)
        ctx.save_for_backward(x)
        ctx.shard = shard
        y_partial = shard(x)                              # локальный вклад
        dist.all_reduce(y_partial, dist.ReduceOp.SUM)     # сумма вкладов -> полный выход
        return y_partial

    @staticmethod
    def backward(ctx, grad_out: torch.Tensor):
        (x,) = ctx.saved_tensors
        shard: TPAttentionShard = ctx.shard
        with torch.enable_grad():
            x_loc = x.detach().requires_grad_(True)
            y_loc = shard(x_loc)                          # без all_reduce здесь
            y_loc.backward(grad_out)                      # локальный ∂L/∂x
            grad_in = x_loc.grad
        dist.all_reduce(grad_in, dist.ReduceOp.SUM)       # суммарный ∂L/∂x
        return None, grad_in


class TPAttention(nn.Module):
    def __init__(self, hidden_size, num_heads, rank, world_size):
        super().__init__()
        self.shard = TPAttentionShard(hidden_size, num_heads, rank, world_size)

    def forward(self, x):
        return ComputeTPAttention.apply(self.shard, x)


@torch.no_grad()
def copy_tp_from_reference(ref: MyLlamaAttention, tp_shard: TPAttentionShard,
                           rank: int, world_size: int):
    """Копируем нужные срезы весов из референса в локальный шард."""
    H = ref.hidden_size
    NH = ref.num_heads
    HD = ref.head_dim
    h_per_rank = NH // world_size
    h_start = rank * h_per_rank
    h_end = (rank + 1) * h_per_rank
    col_start = h_start * HD
    col_end = h_end * HD

    # q,k,v: режем по выходным строкам (наши головы)
    tp_shard.wq.weight.copy_(ref.wq.weight[col_start:col_end])      # [local_hidden, H]
    tp_shard.wk.weight.copy_(ref.wk.weight[col_start:col_end])
    tp_shard.wv.weight.copy_(ref.wv.weight[col_start:col_end])

    # o: режем по входным столбцам (соответствующим нашим головам)
    # ref.wo.weight: [H, H] -> берём [:, col_start:col_end] и транспонируем под форму [local_hidden, H]
    tp_shard.wo.weight.copy_(ref.wo.weight[:, col_start:col_end])


# ------------------------------ main ------------------------------
# ------------------------------ main ------------------------------
if __name__ == "__main__":
    import os

    # 1) Привязываем процесс к конкретному GPU
    local_rank = int(os.environ.get("LOCAL_RANK", 0))
    torch.cuda.set_device(local_rank)
    device = torch.device(f"cuda:{local_rank}")

    # 2) Инициализируем процесс-группу NCCL и укажем device_id (если версия PyTorch поддерживает)
    dist.init_process_group(backend="nccl", device_id=local_rank)
    torch.manual_seed(1337)
    rank, world_size = dist.get_rank(), dist.get_world_size()

    for active_rank in range(world_size):
        dist.barrier()
        if rank != active_rank:
            continue

        # === Референсная (непараллельная) модель ===
        ref_module = MyLlamaAttention(hidden_size=4096, num_heads=32).to(device)

        # входы и (опционально) дополнительные аргументы
        input = torch.randn(1, 128, 4096, device=device, requires_grad=True)
        extra_inputs = dict()

        ref_output = ref_module(input, **extra_inputs)
        ref_output.sum().backward()
        ref_input_grad = input.grad.clone()

        # === TP step 1: модуль, считающий часть голов ===
        tp_module = TPAttention(hidden_size=4096, num_heads=32,
                                rank=rank, world_size=world_size).to(device)

        # === TP step 2: копируем соответствующие веса из референса ===
        with torch.no_grad():
            copy_tp_from_reference(ref_module, tp_module.shard, rank, world_size)

        print(f"Initialized rank={rank}", flush=True)
        del ref_module  # освобождаем RAM для следующего ранга

    # -------- TEST AREA --------
    dist.barrier()  # test 1: forward pass
    tp_input = input.detach().requires_grad_(True)  # уже на device
    tp_output = tp_module(tp_input, **extra_inputs)
    if rank == 0:
        print(f"\nReference outputs ({rank=}):", ref_output.data, flush=True)
    for i in range(world_size):
        dist.barrier()
        if i != rank:
            continue
        print(f"TParallel outputs ({rank=}):", tp_output.data, flush=True)
        assert torch.allclose(tp_output, ref_output, atol=1e-5), f"output mismatch on {rank=}"

    dist.barrier()  # test 2: backward w.r.t. inputs
    assert tp_input.grad is None
    tp_output.sum().backward()
    if rank == 0:
        print(f"\nReference input grad ({rank=}):", ref_input_grad, flush=True)
    for i in range(world_size):
        dist.barrier()
        if i != rank:
            continue
        print(f"TParallel input grad ({rank=}):", tp_input.grad.data, flush=True)
        assert torch.allclose(tp_input.grad, ref_input_grad, atol=1e-4), f"input_grad mismatch on {rank=}"

Overwriting tensor_parallel_attn.py


In [16]:
!OMP_NUM_THREADS=1 torchrun --nproc_per_node=1 tensor_parallel_attn.py

Initialized rank=0

Reference outputs (rank=0): tensor([[[ 0.0140, -0.0784,  0.0378,  ..., -0.0024,  0.0233, -0.0005],
         [ 0.0199, -0.0923,  0.0051,  ..., -0.0100,  0.0396, -0.0070],
         [ 0.0370, -0.0868,  0.0092,  ...,  0.0108,  0.0225, -0.0111],
         ...,
         [ 0.0340, -0.0808,  0.0136,  ...,  0.0074,  0.0249, -0.0051],
         [ 0.0167, -0.0712,  0.0029,  ..., -0.0192,  0.0283, -0.0288],
         [ 0.0059, -0.0795,  0.0153,  ..., -0.0086,  0.0226, -0.0150]]],
       device='cuda:0')
TParallel outputs (rank=0): tensor([[[ 0.0140, -0.0784,  0.0378,  ..., -0.0024,  0.0233, -0.0005],
         [ 0.0199, -0.0923,  0.0051,  ..., -0.0100,  0.0396, -0.0070],
         [ 0.0370, -0.0868,  0.0092,  ...,  0.0108,  0.0225, -0.0111],
         ...,
         [ 0.0340, -0.0808,  0.0136,  ...,  0.0074,  0.0249, -0.0051],
         [ 0.0167, -0.0712,  0.0029,  ..., -0.0192,  0.0283, -0.0288],
         [ 0.0059, -0.0795,  0.0153,  ..., -0.0086,  0.0226, -0.0150]]],
       device='c

__Task 2 (1 point):__ Combine the two previous techniques in one file that parallelizes an actual Llama model and .generates meaningful output. For simplicity, you do not need to partition key-value cache here - only the forward pass itself. We will default to generating tokens with recomputation.

For the sake of formality, your task is to parallelize the following inference code:


In [12]:
import torch
import transformers
MODEL_NAME = "unsloth/Llama-3.2-1B"  # for testing (but not grading!), you may want to use Maykeye/TinyLLama-v0

tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_NAME)
model = transformers.LlamaForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32)  # <-- you are allowed to switch to bf16

prompt = "A quick brown fox"
input_ids = tokenizer(prompt, return_tensors='pt')["input_ids"]
print(end=prompt)
for i in range(5):
  with torch.no_grad():
    new_token = model(input_ids).logits[0, -1].argmax(-1)
    input_ids = torch.cat([input_ids, new_token.view(1, 1)], dim=1)
  print(end=tokenizer.decode(new_token), flush=True)
# pro tip: delete the model or restart session to free RAM for the TP experiments

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/459 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/230 [00:00<?, ?B/s]

A quick brown fox jumps over the lazy dog


**Requirements:** your code must do the following things for the full grade:
- instantiate an actually trained Llama model (Llama 3.2 1B or larger is fine, maykeye is not)
- run forward pass with at least 2 ranks and verify that the logits are close,
- run backward pass w.r.t. non-parallelized input embeddings, verify that the gradients are close,
- perform inference for 10 steps to verify that the model produces meaningful outputs (see below)

You are only required to tensor-parallel-ize the transformer layers. Parallelizing embeddings and logits is optional. If you do choose to parallelize embeddings, we sincerely recommend that you partition across the embedding dim, not across tokens - so that the computation is balanced.

In [ ]:
%%writefile tensor_parallel_llama.py
"""
Task 2: Tensor-Parallel Llama (manual torch.distributed implementation)

Features implemented:
 - Loads reference pretrained model (Llama 3.2 1B) sequentially per rank to reduce peak RAM.
 - Tensor-parallelizes ONLY transformer layers:
     * Attention: heads sharded (column-wise for q/k/v, row-wise for o) + all_reduce on output.
     * MLP: gate & up column shards, down row shard + all_reduce on output.
 - Embeddings, final norm, lm_head are replicated (simpler + OK per assignment spec).
 - Forward correctness: logits (last token) match reference (allclose).
 - Backward correctness: gradient w.r.t. embedding weight matches reference (after all_reduce).
 - 10-step greedy generation demo (recompute, no KV cache) prints output on rank 0.

Switching to bf16: set TORCH_DTYPE below if using GPUs with bf16 support.
To run (CPU example):
   OMP_NUM_THREADS=1 torchrun --nproc_per_node 2 tensor_parallel_llama.py
To run (GPU):
   torchrun --nproc_per_node <N> tensor_parallel_llama.py --backend nccl --dtype bf16
"""
import argparse, os, math
import torch
import torch.nn as nn
import torch.distributed as dist
import torch.nn.functional as F
from transformers import AutoTokenizer, LlamaForCausalLM

# ----------------- Utility linear shards -----------------
class ColumnLinearShard(nn.Module):
    def __init__(self, in_features: int, out_features_total: int, rank: int, world_size: int, bias: bool=False):
        super().__init__()
        assert out_features_total % world_size == 0
        self.out_per_rank = out_features_total // world_size
        self.weight = nn.Parameter(torch.empty(self.out_per_rank, in_features))
        self.bias = nn.Parameter(torch.zeros(self.out_per_rank)) if bias else None
        self.reset_parameters()
    def reset_parameters(self):
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))
        if self.bias is not None:
            fan_in = self.weight.size(1)
            bound = 1 / math.sqrt(fan_in)
            nn.init.uniform_(self.bias, -bound, bound)
    def forward(self, x):
        return F.linear(x, self.weight, self.bias)

class RowLinearShard(nn.Module):
    def __init__(self, in_features_total: int, out_features: int, rank: int, world_size: int, bias: bool=False):
        super().__init__()
        assert in_features_total % world_size == 0
        self.in_per_rank = in_features_total // world_size
        self.weight = nn.Parameter(torch.empty(out_features, self.in_per_rank))
        self.bias = nn.Parameter(torch.zeros(out_features)) if bias else None
        self.reset_parameters()
    def reset_parameters(self):
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))
        if self.bias is not None:
            fan_in = self.weight.size(1)
            bound = 1 / math.sqrt(fan_in)
            nn.init.uniform_(self.bias, -bound, bound)
    def forward(self, x_partial):  # x_partial (..., in_per_rank)
        return F.linear(x_partial, self.weight, self.bias)

# ----------------- Attention (head sharding) -----------------
class TPAttention(nn.Module):
    def __init__(self, hidden_size: int, num_heads: int, rank: int, world_size: int):
        super().__init__()
        assert num_heads % world_size == 0
        self.hidden_size = hidden_size
        self.num_heads = num_heads
        self.rank = rank
        self.world_size = world_size
        self.local_heads = num_heads // world_size
        self.head_dim = hidden_size // num_heads
        local_hidden = self.local_heads * self.head_dim
        self.wq = ColumnLinearShard(hidden_size, hidden_size, rank, world_size)
        self.wk = ColumnLinearShard(hidden_size, hidden_size, rank, world_size)
        self.wv = ColumnLinearShard(hidden_size, hidden_size, rank, world_size)
        self.wo = RowLinearShard(hidden_size, hidden_size, rank, world_size)
    def _reshape(self, x_local):  # (B,S,local_hidden)->(B, h_loc, S, D)
        B,S,LH = x_local.shape
        return x_local.view(B,S,self.local_heads,self.head_dim).permute(0,2,1,3)
    def forward(self, x):
        q = self._reshape(self.wq(x))
        k = self._reshape(self.wk(x))
        v = self._reshape(self.wv(x))
        attn = F.scaled_dot_product_attention(q,k,v)  # (B,h_loc,S,D)
        B,h,S,D = attn.shape
        attn = attn.permute(0,2,1,3).reshape(B,S,h*D)
        out_partial = self.wo(attn)  # (B,S,H) partial rows
        dist.all_reduce(out_partial, op=dist.ReduceOp.SUM)
        return out_partial

# ----------------- MLP (unit sharding) -----------------
class TPMLP(nn.Module):
    def __init__(self, hidden_size: int, intermediate_size: int, rank: int, world_size: int):
        super().__init__()
        self.gate = ColumnLinearShard(hidden_size, intermediate_size, rank, world_size)
        self.up = ColumnLinearShard(hidden_size, intermediate_size, rank, world_size)
        self.down = RowLinearShard(intermediate_size, hidden_size, rank, world_size)
    def forward(self, x):
        g = torch.silu(self.gate(x))
        u = self.up(x)
        inter_partial = g * u  # (B,S,inter_per_rank)
        down_partial = self.down(inter_partial)
        dist.all_reduce(down_partial, op=dist.ReduceOp.SUM)
        return down_partial

# ----------------- TP Transformer Block -----------------
class TPBlock(nn.Module):
    def __init__(self, ref_block, rank: int, world_size: int):
        super().__init__()
        hs = ref_block.input_layernorm.normalized_shape[0]
        self.rms1 = nn.RMSNorm(hs)
        self.rms2 = nn.RMSNorm(hs)
        self.attn = TPAttention(hs, ref_block.self_attn.num_heads, rank, world_size)
        self.mlp = TPMLP(hs, ref_block.mlp.gate_proj.out_features, rank, world_size)
    def forward(self, x):
        x = x + self.attn(self.rms1(x))
        x = x + self.mlp(self.rms2(x))
        return x

# ----------------- Full TP Llama (layers only) -----------------
class TensorParallelLlama(nn.Module):
    def __init__(self, ref_model: LlamaForCausalLM, rank: int, world_size: int):
        super().__init__()
        self.config = ref_model.config
        self.embed = nn.Embedding(ref_model.model.embed_tokens.num_embeddings, ref_model.model.embed_tokens.embedding_dim)
        self.layers = nn.ModuleList([TPBlock(ref_layer, rank, world_size) for ref_layer in ref_model.model.layers])
        self.final_norm = nn.RMSNorm(ref_model.model.norm.normalized_shape[0])
        self.lm_head = nn.Linear(ref_model.lm_head.in_features, ref_model.lm_head.out_features, bias=False)
        self.rank = rank
        self.world_size = world_size
    def forward(self, input_ids):
        x = self.embed(input_ids)
        for layer in self.layers:
            x = layer(x)
        x = self.final_norm(x)
        logits = self.lm_head(x)
        return logits

# ----------------- Weight copy helpers -----------------
@torch.no_grad()
def copy_attention(ref_attn, tp_attn: TPAttention, rank: int, world_size: int):
    num_heads = ref_attn.num_heads
    head_dim = ref_attn.head_dim
    heads_per_rank = num_heads // world_size
    start = rank * heads_per_rank * head_dim
    end = (rank+1)*heads_per_rank*head_dim
    for shard_lin, ref_lin in [ (tp_attn.wq, ref_attn.q_proj), (tp_attn.wk, ref_attn.k_proj), (tp_attn.wv, ref_attn.v_proj) ]:
        shard_lin.weight.copy_(ref_lin.weight[start:end])
    # o_proj row-shard: take columns start:end
    tp_attn.wo.weight.copy_(ref_attn.o_proj.weight[:, start:end])

@torch.no_grad()
def copy_mlp(ref_mlp, tp_mlp: TPMLP, rank: int, world_size: int):
    inter = ref_mlp.gate_proj.out_features
    part = inter // world_size
    s = rank * part; e = (rank+1)*part
    tp_mlp.gate.weight.copy_(ref_mlp.gate_proj.weight[s:e])
    tp_mlp.up.weight.copy_(ref_mlp.up_proj.weight[s:e])
    tp_mlp.down.weight.copy_(ref_mlp.down_proj.weight[:, s:e])

@torch.no_grad()
def copy_block(ref_block, tp_block: TPBlock, rank: int, world_size: int):
    tp_block.rms1.load_state_dict(ref_block.input_layernorm.state_dict())
    tp_block.rms2.load_state_dict(ref_block.post_attention_layernorm.state_dict())
    copy_attention(ref_block.self_attn, tp_block.attn, rank, world_size)
    copy_mlp(ref_block.mlp, tp_block.mlp, rank, world_size)

@torch.no_grad()
def copy_full_model(ref_model: LlamaForCausalLM, tp_model: TensorParallelLlama, rank: int, world_size: int):
    tp_model.embed.weight.copy_(ref_model.model.embed_tokens.weight)
    tp_model.final_norm.load_state_dict(ref_model.model.norm.state_dict())
    tp_model.lm_head.weight.copy_(ref_model.lm_head.weight)
    for rb, tb in zip(ref_model.model.layers, tp_model.layers):
        copy_block(rb, tb, rank, world_size)

# ----------------- Generation (greedy) -----------------
@torch.no_grad()
def tp_generate(tp_model: TensorParallelLlama, tokenizer, input_ids: torch.Tensor, steps: int=10):
    for _ in range(steps):
        logits = tp_model(input_ids)
        next_token = logits[:, -1].argmax(-1, keepdim=True)
        input_ids = torch.cat([input_ids, next_token], dim=1)
        if tp_model.rank == 0:
            print(tokenizer.decode(next_token[0]), end="", flush=True)
    if tp_model.rank == 0:
        print()
    return input_ids

# ----------------- Main -----------------

def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument('--model', default='unsloth/Llama-3.2-1B')
    p.add_argument('--backend', default='gloo')  # nccl for multi-gpu
    p.add_argument('--dtype', default='fp32', choices=['fp32','bf16','fp16'])
    p.add_argument('--gen-steps', type=int, default=10)
    return p.parse_args()

def str_to_dtype(s):
    if s=='fp32': return torch.float32
    if s=='bf16': return torch.bfloat16
    if s=='fp16': return torch.float16
    raise ValueError(s)

def main():
    args = parse_args()
    dist.init_process_group(args.backend)
    rank = dist.get_rank(); world_size = dist.get_world_size()
    torch.manual_seed(1337)
    dtype = str_to_dtype(args.dtype)

    # Sequential load to reduce peak memory
    tokenizer = None
    ref_model = None
    for r in range(world_size):
        dist.barrier()
        if r != rank: continue
        if rank == 0:
            print(f"Loading reference model {args.model} (dtype={dtype}) ...")
        ref_model = LlamaForCausalLM.from_pretrained(args.model, torch_dtype=dtype)
        tokenizer = AutoTokenizer.from_pretrained(args.model)
        if tokenizer.pad_token_id is None:
            tokenizer.pad_token_id = tokenizer.eos_token_id
        if rank == 0:
            print("Reference model loaded.")

    # Broadcast that model is ready (lightweight barrier already done above)
    dist.barrier()

    # Build TP wrapper & copy weights
    tp_model = TensorParallelLlama(ref_model, rank, world_size).to(dtype)
    copy_full_model(ref_model, tp_model, rank, world_size)

    # Prepare prompt
    prompt = "A quick brown fox"
    input_ids = tokenizer(prompt, return_tensors='pt')["input_ids"]

    # ---- Forward correctness (logits) ----
    ref_input_ids = input_ids.clone().requires_grad_(True)
    ref_logits = ref_model(ref_input_ids).logits  # (1, T, V)
    # Only need last position for check (but we'll compare full tensor)
    tp_logits = tp_model(input_ids.clone())
    # All ranks have same (reduced) logits because each layer reduces; just compare on rank 0
    if rank == 0:
        max_diff = (tp_logits - ref_logits).abs().max().item()
        print(f"[Check] Logits max abs diff: {max_diff:.3e}")
        assert torch.allclose(tp_logits, ref_logits, atol=1e-4), "Logits mismatch"

    # ---- Backward correctness (embedding grad) ----
    ref_loss = ref_logits[:, -1].sum()
    ref_loss.backward()
    ref_emb_grad = ref_model.model.embed_tokens.weight.grad.clone()

    # Enable grad on replicated embedding of TP model
    tp_model.embed.weight.requires_grad_(True)
    tp_loss = tp_logits[:, -1].sum()
    tp_loss.backward()
    # Reduce embedding grad across ranks (replicated param)
    dist.all_reduce(tp_model.embed.weight.grad, op=dist.ReduceOp.SUM)

    if rank == 0:
        grad_diff = (tp_model.embed.weight.grad - ref_emb_grad).abs().max().item()
        print(f"[Check] Embedding grad max abs diff: {grad_diff:.3e}")
        assert grad_diff < 1e-4, "Embedding grad mismatch"
        print("Forward & backward correctness PASSED.")

    # ---- Generation demo ----
    if rank == 0:
        print("TP generation:", end=" ")
        print(prompt, end="")
    tp_generate(tp_model, tokenizer, input_ids, steps=args.gen_steps)

    if rank == 0:
        print("Task 2 complete on all ranks.")

    # Free reference model early (optional)
    del ref_model
    dist.barrier()

if __name__ == "__main__":
    main()


In [ ]:
<<... and a dedicated cell to show off that it works>>




### Using [`torch.distributed.tensor`](https://pytorch.org/docs/stable/distributed.tensor.html)

PyTorch has an in-built functionality called [DTensor](https://pytorch.org/docs/stable/distributed.tensor.html), designed to help implementing tensor-level parallelism with various sharding strategies. This includes Tensor parallelism itself, as well as other techniques such as Sequence Parallelism, as they are both, essentially, parallelism across different tensor dimensions.

__Task 3 (1 point):__ Your next task will be to replicate your previous code (llama inference) using DTensor instead of manual AllReduce. We recommend you start by skimming the [documentation for DTensor](https://pytorch.org/docs/stable/distributed.tensor.html) to learn the interface and [the minimal example](https://github.com/pytorch/examples/blob/main/distributed/tensor_parallelism/tensor_parallel_example.py) to learn how to put the pieces together.


We recommend that you dedicate some time to learn and play with it before you proceed to parallelize Llama.

The main objective is the same as in the previous task - run .generate with DTensor - and then compare it against the manual implementation. **Please report at least some speed comparison for forward and backward passes between this and the previous task.** If absolutely impossible (e.g. you don't have multiple gpus), we can accept a fallback assignment of implementing basic training: overfit the model to a single batch (like task 5 below) and demonstrate that it works - if you choose this option, say so in bold, large-font letters somewhere where the grader can see.

But first, here's a quick demo of using DTensor for simple matrix multiplication - meant as a testbed for your experiments.

In [1]:
%%writefile tensor_parallel_mlp_dtensor.py
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributed as dist
from torch.distributed.device_mesh import init_device_mesh
from torch.distributed.tensor import DTensor, DeviceMesh, Replicate, Shard
import torch.distributed.tensor.parallel as tp


class LlamaMLP(nn.Module):  # same module, but with smaller dims for quick prototyping
    def __init__(self, hidden_size: int = 1024, intermediate_size: int = 4096):
        super().__init__()
        self.gate_proj = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.up_proj = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.down_proj = nn.Linear(intermediate_size, hidden_size, bias=False)

    def forward(self, input):
        return self.down_proj(F.silu(self.gate_proj(input)) * self.up_proj(input))


if __name__ == "__main__":
    dist.init_process_group("gloo")  # use nccl for cuda devices
    torch.manual_seed(1337)          # init weights equally on all ranks
    rank, world_size = dist.get_rank(), dist.get_world_size()

    # Initialize device mesh for tensor parallelism
    device_mesh = init_device_mesh(device_type="cpu", mesh_shape=(world_size,))  # use "cuda" for GPU

    # Create reference module for comparison
    ref_module = nn.Sequential(nn.RMSNorm(1024), LlamaMLP())

    input = torch.randn(1, 1024, requires_grad=True)
    ref_output = ref_module(input)
    ref_output.sum().backward()
    ref_input_grad = input.grad.clone()

    # Create tensor parallel module (we wrap ref_module instead of copying)
    tp_module = tp.parallelize_module(
        ref_module,
        device_mesh,
        parallelize_plan={  # define parallelism type for each module
            # up_proj and gate_proj are column-wise parallel (sharded across outputs);
            "1.up_proj": tp.ColwiseParallel(),
            "1.gate_proj": tp.ColwiseParallel(),
            # down_proj is row-wise parallel (sharded across input dim)
            "1.down_proj": tp.RowwiseParallel(),
          },  # note: RMSNorm is simply replicated across all devices - hence, we skip it
    )
    if rank == 0:  # Note: no need to copy weight chunks manually: DTensor handles parameter sharding for us
      for name, param in tp_module.named_parameters():
        print(f"{name=},\ttype={type(param.data)}\tglobal shape={param.shape},\tlocal shape={param._local_tensor.shape if hasattr(param, '_local_tensor') else param.shape}")

    dist.barrier()  # Test forward and backward pass with Tensor Parallelism
    tp_input = input.detach().requires_grad_(True)
    tp_output = tp_module(tp_input)
    tp_output.sum().backward()
    tp_output = tp_output.trigger_wait()  # convert from AsyncCollectiveTensor to regular torch tensor
    if rank == 0:
        print(f"\nReference outputs ({rank=}):", ref_output.data, flush=True)
        print(f"TParallel outputs ({rank=}):", tp_output.data, flush=True)
        print(f"\nReference input grad ({rank=}):", ref_input_grad, flush=True)
        print(f"TParallel input grad ({rank=}):", tp_input.grad, flush=True)
    dist.barrier()
    assert torch.allclose(tp_output, ref_output, atol=1e-6), f"output mismatch on {rank=}"
    assert torch.allclose(tp_input.grad, ref_input_grad, atol=1e-6), f"input_grad mismatch on {rank=}"
    print(end=f"Tests passed ({rank=})\n", flush=True); dist.barrier()

# fun fact: 90% of the code above was generated by grok-3 for prompt "Please rewrite the following code using torch.distributed.tensor ```python <paste MLP code here>```"
# the remaining 10% are nasty bugfixes that took 99% of assignment preparation time. Do not trust the shogoths yet :)

Writing tensor_parallel_mlp_dtensor.py


In [2]:
!OMP_NUM_THREADS=1 torchrun --nproc_per_node 2 tensor_parallel_mlp_dtensor.py

[Gloo] Rank 1 is connected to 1 peer ranks. Expected number of connected peer ranks is : 1
[Gloo] Rank 0 is connected to 1 peer ranks. Expected number of connected peer ranks is : 1
name='0.weight',	type=<class 'torch.Tensor'>	global shape=torch.Size([1024]),	local shape=torch.Size([1024])
name='1.gate_proj.weight',	type=<class 'torch.distributed.tensor.DTensor'>	global shape=torch.Size([4096, 1024]),	local shape=torch.Size([2048, 1024])
name='1.up_proj.weight',	type=<class 'torch.distributed.tensor.DTensor'>	global shape=torch.Size([4096, 1024]),	local shape=torch.Size([2048, 1024])
name='1.down_proj.weight',	type=<class 'torch.distributed.tensor.DTensor'>	global shape=torch.Size([1024, 4096]),	local shape=torch.Size([1024, 2048])

Reference outputs (rank=0): tensor([[ 0.0102,  0.0432, -0.0467,  ...,  0.0798, -0.0179,  0.0527]])
TParallel outputs (rank=0): tensor([[ 0.0102,  0.0432, -0.0467,  ...,  0.0798, -0.0179,  0.0527]])

Reference input grad (rank=0): tensor([[ 0.1543, -0.0858, 

In [1]:
%%writefile tensor_parallel_llama_dtensor.py
"""
Task 3: DTensor-based Tensor Parallel Llama Inference + Speed Comparison

We re-implement Task 2 (manual all_reduce tensor parallel) using torch.distributed.tensor (DTensor) APIs.
Goals:
 1. Shard transformer layer parameters (attention q/k/v col-wise, o row-wise; MLP gate/up col-wise, down row-wise) using tp.* helpers.
 2. Replicate embeddings, final norm, lm_head (simpler per assignment).
 3. Run correctness checks against full reference model (logits + embedding gradient) on rank 0.
 4. Provide a simple timing comparison vs manual implementation (user should have run Task 2 first). We measure forward+backward latency for a few warmup + measured iterations.
 5. Run 10-step greedy generation with DTensor model (recompute, no KV cache).

Assumptions:
 - world_size divides num_heads and intermediate_size.
 - Backend: gloo (CPU) or nccl (GPU). Use bf16/fp16 if needed for memory.
 - PyTorch version supports distributed.tensor.parallel (>=2.3+ typically).

How to run (CPU example, 2 ranks):
   OMP_NUM_THREADS=1 torchrun --nproc_per_node 2 tensor_parallel_llama_dtensor.py --model unsloth/Llama-3.2-1B --backend gloo --dtype fp32
GPU example:
   torchrun --nproc_per_node 2 tensor_parallel_llama_dtensor.py --backend nccl --dtype bf16
"""
import argparse, time, math
import torch
import torch.nn as nn
import torch.distributed as dist
import torch.nn.functional as F
from transformers import LlamaForCausalLM, AutoTokenizer

from torch.distributed.device_mesh import init_device_mesh
import torch.distributed.tensor.parallel as tp

# ---------------- Argument parsing ----------------

def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument('--model', default='unsloth/Llama-3.2-1B')
    p.add_argument('--backend', default='gloo')
    p.add_argument('--dtype', default='fp32', choices=['fp32','bf16','fp16'])
    p.add_argument('--gen-steps', type=int, default=10)
    p.add_argument('--warmup', type=int, default=2)
    p.add_argument('--iters', type=int, default=5)
    return p.parse_args()

def str_to_dtype(s):
    if s=='fp32': return torch.float32
    if s=='bf16': return torch.bfloat16
    if s=='fp16': return torch.float16
    raise ValueError(s)

# ------------- DTensor parallelization helper -------------

def parallelize_llama_layers(ref_model: LlamaForCausalLM, device_mesh):
    """Return a module whose transformer layers are tensor-parallel via DTensor.
    Strategy (mirrors manual TP):
      gate_proj, up_proj -> ColwiseParallel
      down_proj -> RowwiseParallel
      q_proj, k_proj, v_proj -> ColwiseParallel
      o_proj -> RowwiseParallel
    RMSNorms remain replicated.
    """
    # We apply plan onto the submodules referencing original modules (in-place re-sharding)
    parallelize_plan = {}
    for layer_idx, layer in enumerate(ref_model.model.layers):
        prefix = f"model.layers.{layer_idx}"
        # attention
        parallelize_plan[f"{prefix}.self_attn.q_proj"] = tp.ColwiseParallel()
        parallelize_plan[f"{prefix}.self_attn.k_proj"] = tp.ColwiseParallel()
        parallelize_plan[f"{prefix}.self_attn.v_proj"] = tp.ColwiseParallel()
        parallelize_plan[f"{prefix}.self_attn.o_proj"] = tp.RowwiseParallel()
        # mlp
        parallelize_plan[f"{prefix}.mlp.gate_proj"] = tp.ColwiseParallel()
        parallelize_plan[f"{prefix}.mlp.up_proj"] = tp.ColwiseParallel()
        parallelize_plan[f"{prefix}.mlp.down_proj"] = tp.RowwiseParallel()
    # NOTE: embed_tokens, final norm, lm_head left replicated
    tp_module = tp.parallelize_module(ref_model, device_mesh, parallelize_plan=parallelize_plan)
    return tp_module

# ------------- Timing utilities -------------

def measure_latency(model, input_ids, steps_fwd_bwd=1, backward=True):
    torch.cuda.synchronize() if torch.cuda.is_available() else None
    start = time.perf_counter()
    for _ in range(steps_fwd_bwd):
        out = model(input_ids).logits
        loss = out[:, -1].sum()
        if backward:
            loss.backward()
            for p in model.parameters():
                if p.grad is not None:
                    p.grad.zero_()
    torch.cuda.synchronize() if torch.cuda.is_available() else None
    return (time.perf_counter() - start) / steps_fwd_bwd

# ------------- Generation (no KV cache) -------------
@torch.no_grad()
def generate(model, tokenizer, input_ids, steps=10):
    for _ in range(steps):
        logits = model(input_ids).logits
        next_token = logits[:, -1].argmax(-1, keepdim=True)
        input_ids = torch.cat([input_ids, next_token], dim=1)
        if dist.get_rank() == 0:
            print(tokenizer.decode(next_token[0]), end="", flush=True)
    if dist.get_rank() == 0:
        print()
    return input_ids

# ------------- Main -------------

def main():
    args = parse_args()
    dist.init_process_group(args.backend)
    rank = dist.get_rank(); world_size = dist.get_world_size()
    dtype = str_to_dtype(args.dtype)
    torch.manual_seed(1234)

    if rank == 0:
        print(f"[Init] Loading reference model: {args.model} (dtype={dtype}, world_size={world_size})")
    # Sequential load to save memory (barrier per rank)
    model = None; tokenizer = None
    for r in range(world_size):
        dist.barrier()
        if r != rank: continue
        model = LlamaForCausalLM.from_pretrained(args.model, torch_dtype=dtype)
        tokenizer = AutoTokenizer.from_pretrained(args.model)
        if tokenizer.pad_token_id is None:
            tokenizer.pad_token_id = tokenizer.eos_token_id
        if rank == 0:
            print("[Init] Reference model loaded.")

    dist.barrier()

    # Create device mesh (cpu or cuda)
    device_type = 'cuda' if torch.cuda.is_available() else 'cpu'
    device_mesh = init_device_mesh(device_type=device_type, mesh_shape=(world_size,))


    # Prepare input
    prompt = "A quick brown fox"
    input_ids = tokenizer(prompt, return_tensors='pt')["input_ids"].to(device_type)
    input_ids_ref = input_ids.clone()#.requires_grad_(True)
    model.zero_grad(set_to_none=True)

    # Reference forward/backward (rank 0 only) for correctness
    if rank == 0:
        ref_out = model(input_ids_ref).logits
        ref_loss = ref_out[:, -1].sum(); ref_loss.backward()
        ref_emb_grad = model.model.embed_tokens.weight.grad.clone()
        ref_logits_detached = ref_out.detach()
    else:
        ref_logits_detached = torch.zeros((1, input_ids.shape[1], model.lm_head.out_features), dtype=dtype, device=device_type)
        ref_emb_grad = torch.zeros_like(model.model.embed_tokens.weight)

    # Broadcast reference tensors to all ranks for comparison
    dist.broadcast(ref_logits_detached, src=0)
    dist.broadcast(ref_emb_grad, src=0)


    # Parallelize in-place
    if rank == 0:
        print("[TP] Applying DTensor sharding plan to transformer layers ...")
    tp_model = parallelize_llama_layers(model, device_mesh)

    # DTensor forward/backward
    dt_out = tp_model(input_ids).logits
    dt_loss = dt_out[:, -1].sum(); dt_loss.backward()

    # Embedding grad (replicated) should match
    emb_grad = tp_model.model.embed_tokens.weight.grad
    # All-reduce grad to mirror reference accumulation semantics (DTensor may have already placed grads properly; safe to sum)
    dist.all_reduce(emb_grad, op=dist.ReduceOp.SUM)

    if rank == 0:
        logits_diff = (dt_out - ref_logits_detached).abs().max().item()
        grad_diff = (emb_grad - ref_emb_grad).abs().max().item()
        print(f"[Check] logits max|diff| = {logits_diff:.3e}")
        print(f"[Check] embed grad max|diff| = {grad_diff:.3e}")
        assert logits_diff < 1e-4, "Logits mismatch"
        assert grad_diff < 1e-4, "Embedding grad mismatch"
        print("[Check] Correctness PASSED.")

    # Timing benchmark
    dist.barrier()
    if rank == 0:
        print("[Bench] Measuring DTensor forward+backward latency ...")
    # Warmup
    for _ in range(args.warmup):
        _ = tp_model(input_ids).logits[:, -1].sum().backward()
        for p in tp_model.parameters():
            if p.grad is not None: p.grad.zero_()
    dist.barrier()
    t_dt = measure_latency(tp_model, input_ids, steps_fwd_bwd=args.iters, backward=True)

    # Dummy manual baseline placeholder (user may record from Task 2 run)
    # We log dtensor time; user can copy manual time from Task2 run to compare.
    if rank == 0:
        print(f"[Bench] DTensor avg fwd+bwd time over {args.iters} iters: {t_dt:.4f}s (record manual TP separately)")

    # Generation
    if rank == 0:
        print("[Gen] DTensor generation:", end=" ")
        print(prompt, end="")
    generate(tp_model, tokenizer, input_ids, steps=args.gen_steps)

    if rank == 0:
        print("Task 3 DTensor implementation complete.")

    dist.barrier()
    del model

if __name__ == '__main__':
    main()


Writing tensor_parallel_llama_dtensor.py


In [3]:
!OMP_NUM_THREADS=1 torchrun --nproc_per_node 2 tensor_parallel_llama_dtensor.py --backend gloo --dtype bf16 --gen-steps 10 --warmup 1 --iters 3
# GPU example:
# torchrun --nproc_per_node 2 tensor_parallel_llama_dtensor.py --backend nccl --dtype bf16 --gen-steps 10 --warmup 2 --iters 5

2025-09-21 12:05:40.856463: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-09-21 12:05:40.856559: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1758456341.101973    7701 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1758456341.101980    7700 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1758456341.174096    7700 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
E0000 00:00:1758456341.174096    7701 cuda_blas.cc:1

```

```

```

```

```

```

```

```

```

```

```

```

```

```

```

```



### Sequence Parallelism with Ulysses


Now let's parallelize the other way - across the sequence dimension. To showcase why this is necessary, our main task will be to parallelize LLM fine-tuning over a very long sequence. The way you do this, of course, is through Sequence Parallelism. You can implement naive [sequence parallelism](https://arxiv.org/abs/2205.05198), similar to [DeepSpeed Ulysses](https://arxiv.org/pdf/2309.14509) (n.b.: not the first work to do this).

![figure-from-paper](https://ar5iv.labs.arxiv.org/html/2309.14509/assets/figs/image3.png)


Here's the short version:
- All weights are replicated between ranks (optionally: FSDP)
- Each rank holds a subset of sequence tokens
- Embeddings, logits, normalizations, MLP all apply independently to token shards
- The multi-head attention is the only layer that gets special treatment
    - First, apply QKV projections to local tokens, as in data-parallel training;
    - Then re-shard so that each rank holds a **subset of heads** across **all tokens**;
    - Compute the attention ''core'' (RoPE and F.scaled_dot_product_attention) for its chunk of heads independently;
    - Re-shard outputs again so that each rank concatenates **all heads**, but only for its **subset of tokens**;
    - Apply the output ("O") projection to your local tokens again.
- This approach *may* be combined with tensor parallelism, but this is an advanced technique that you don't have to implement.


__You have a choice__ between two options on how to implement it: either manually with torch.distributed like in task 2, or using the DTensor route like in task 3. We provide some tips for both tasks.


**Option A. with raw `torch.distirbuted`:**
- Use [`dist.all_to_all`](https://pytorch.org/docs/stable/distributed.html#torch.distributed.all_to_all) to switch between per-token and per-head sharding without materializing the full tensor on any device;
- Wrap the model with [`DistributedDataParallel`](https://pytorch.org/tutorials/intermediate/ddp_tutorial.html) or [`FullyShardedDataParallel`](https://pytorch.org/docs/stable/fsdp.html) so that fine-tuning synchronizes trainable parameters. Note that using FSDP for parameter-efficient fine-tuning can be tricky: we recommend you either wrap **trainable modules** with separate FSDP sub-instances via auto_wrap_policy - or simply use DDP instead of FSDP.

**Option B. with `DTensor`:**
- We recommend you first skim the official [tutorial](https://pytorch.org/tutorials/intermediate/TP_tutorial.html) on applying Tensor Parallelism (sic.) - or browse the [TorchTitan's version](https://github.com/pytorch/torchtitan/blob/82afc842e303e49d1a137fc7ea48291a57f72d5d/torchtitan/models/llama/parallelize_llama.py) of it.
- Note that there is a [`SequenceParallel`](https://pytorch.org/docs/stable/distributed.tensor.parallel.html#torch.distributed.tensor.parallel.SequenceParallel) class in torch.distributed.tensor.parallel` - **however, it does not magick the sequence parallelism for you** - it is only meant for small layers (e.g. normalization). You still need to do the sharding in self-attention!

For the sake of formality, here's an example script you need to parallelize:

In [ ]:
import torch
import transformers
import peft
MODEL_NAME = "unsloth/Llama-3.2-1B"  # for testing (but not grading!), you may want to use Maykeye/TinyLLama-v0
SEQUENCE_LENGTH = 128                # IMPORTANT!!! you need to increase this parameter! Look for the maximum sequence length on one and multiple GPUs

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_NAME)
model = transformers.LlamaForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.bfloat16).to(device)

for param in model.parameters():
  param.required_grad = False
model.gradient_checkpointing_enable()
model.enable_input_require_grads()

model = peft.get_peft_model(model, peft.PromptTuningConfig(task_type=peft.TaskType.CAUSAL_LM, num_virtual_tokens=32))
assert any(param.requires_grad for param in model.parameters()), "No trainable parameters - did you enable PEFT?"

!wget -q https://www.gutenberg.org/cache/epub/4300/pg4300.txt -O ulysses.txt  # ... or use any other text of your choosing
input_ids = tokenizer(open("ulysses.txt").read(), return_tensors='pt')['input_ids']
print(f"Cropping {input_ids.shape[1]=} to {SEQUENCE_LENGTH} tokens")
input_ids, labels = input_ids[:, :SEQUENCE_LENGTH], input_ids[:, 1:SEQUENCE_LENGTH + 1]

trainable_parameters = {p for p in model.parameters() if p.requires_grad}
print(f"Parameters: {sum(map(torch.Tensor.numel, trainable_parameters))} trainable / {sum(map(torch.Tensor.numel, model.parameters()))} total")
opt = torch.optim.Adam(trainable_parameters)
for i in range(10):
  loss = model(input_ids=input_ids.to(device), labels=labels.to(device)).loss
  opt.zero_grad()
  loss.backward()
  opt.step()
  print(f"{i=}\t{loss.item()=}")

# pro tip: delete the model or restart session to free RAM for the TP experiments

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/50.6k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/459 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/935 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/230 [00:00<?, ?B/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (397368 > 131072). Running this sequence through the model will result in indexing errors


Cropping input_ids.shape[1]=397368 to 128 tokens
Parameters: 65536 trainable / 1235879936 total
i=0	loss.item()=8.582364082336426
i=1	loss.item()=8.413138389587402
i=2	loss.item()=8.251019477844238
i=3	loss.item()=8.113922119140625
i=4	loss.item()=8.001195907592773
i=5	loss.item()=7.875072002410889
i=6	loss.item()=7.784201145172119
i=7	loss.item()=7.691455364227295
i=8	loss.item()=7.621612548828125
i=9	loss.item()=7.547983169555664


__Task 4 (1 point):__ before you do training, let's first parallelize a single forward pass. Implement sharding with the same interface you used in tasks 2 (or 3 if you use DTensor), but this time, parallelize across the sequence dimension. Note: if you are running out of (V)RAM, load the 1B model in half precision and disable gradients for all weights except the first (few) layers.



In [2]:
%%writefile sequence_parallel_forward.py
import os, argparse
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributed as dist
from transformers import LlamaForCausalLM, AutoTokenizer, LlamaConfig
from transformers.models.llama.modeling_llama import LlamaRotaryEmbedding, apply_rotary_pos_emb

# -------- helpers --------
def split_sequence(input_ids, rank, world):
    B, S = input_ids.shape
    assert S % world == 0, "S must be divisible by world"
    S_local = S // world
    sl = slice(rank * S_local, (rank+1)*S_local)
    return input_ids[:, sl], sl, S_local

def tokens_to_heads_alltoall(t_local, world, H_total, Hloc):
    """
    t_local: (B, N_local, H_total) on each rank (token-shard)
    Return th_local: (B, N, Hloc) on each rank (head-shard)
    """
    B, N_local, _ = t_local.shape
    N = N_local * world
    # reshape to (B, N_local, world, Hloc) — split H into head-shards
    t = t_local.view(B, N_local, world, Hloc)
    send = list(t.unbind(dim=2))  # list of world tensors: (B, N_local, Hloc)

    recv = [torch.empty_like(send[0]) for _ in range(world)]
    dist.all_to_all(recv, send)
    # concat along tokens -> (B, N, Hloc)
    th_local = torch.cat(recv, dim=1)
    return th_local

def heads_to_tokens_alltoall(th_local, world, H_total, Hloc):
    """
    th_local: (B, N, Hloc) on each rank (head-shard)
    Return t_local: (B, N_local, H_total) on each rank (token-shard)
    """
    B, N, _ = th_local.shape
    N_local = N // world
    # split along tokens -> list world of (B, N_local, Hloc)
    chunks = list(th_local.split(N_local, dim=1))
    recv = [torch.empty_like(chunks[0]) for _ in range(world)]
    dist.all_to_all(recv, chunks)  # now each rank holds (B, N_local, Hloc) from every rank
    # concat along head-shards -> (B, N_local, H_total)
    t_local = torch.cat(recv, dim=2)
    return t_local

# -------- attention with SP + RoPE + two all_to_all --------
class SeqParAttention(nn.Module):
    def __init__(self, ref_attn, config, rank, world):
        super().__init__()
        self.hidden_size = ref_attn.q_proj.in_features
        print(config)
        self.num_heads = getattr(config, "num_attention_heads", None)
        self.head_dim = self.hidden_size // self.num_heads
        assert self.hidden_size % self.num_heads == 0
        assert self.num_heads % world == 0

        self.rank = rank
        self.world = world
        self.hloc = self.num_heads // world  # local heads per rank

        # replicate weights (teaching version); можно зашардить как TP
        self.q_proj = nn.Linear(self.hidden_size, self.hidden_size, bias=False)
        self.k_proj = nn.Linear(self.hidden_size, self.hidden_size, bias=False)
        self.v_proj = nn.Linear(self.hidden_size, self.hidden_size, bias=False)
        self.o_proj = nn.Linear(self.hidden_size, self.hidden_size, bias=False)

        # RoPE
        self.rope = LlamaRotaryEmbedding(config)

    def forward(self, x_local, position_ids):
        """
        x_local: (B, N_local, H)
        position_ids: (B, N) full positions for the whole sequence
        """
        B, N_local, H = x_local.shape
        world = self.world
        Hloc_vec = self.hloc * self.head_dim  # local head hidden

        # local QKV on token-shard
        q = self.q_proj(x_local)  # (B, N_local, H)
        k = self.k_proj(x_local)
        v = self.v_proj(x_local)

        # token->head shard (without materializing full S on each rank)
        q_h = tokens_to_heads_alltoall(q, world, H, Hloc_vec)  # (B, N, Hloc_vec)
        k_h = tokens_to_heads_alltoall(k, world, H, Hloc_vec)
        v_h = tokens_to_heads_alltoall(v, world, H, Hloc_vec)

        # reshape to (B, hloc, N, D)
        def to_heads(t):  # t: (B, N, Hloc*D)
            B_, N_, _ = t.shape
            return t.view(B_, N_, self.hloc, self.head_dim).permute(0,2,1,3).contiguous()
        qh = to_heads(q_h)
        kh = to_heads(k_h)
        vh = to_heads(v_h)  # (B, hloc, N, D)

        # apply RoPE to Q/K (need cos,sin for all positions)
        # position_ids: (B, N) — одинаковый на всех рангах
        cos, sin = self.rope(vh.transpose(1,2), position_ids)  # HF API: pass shape (B,N,*) to get cos/sin
        # apply_rotary_pos_emb expects (..., seq_len, dim)
        qh = apply_rotary_pos_emb(qh.transpose(1,2), cos, sin, position_ids).transpose(1,2)
        kh = apply_rotary_pos_emb(kh.transpose(1,2), cos, sin, position_ids).transpose(1,2)

        # local SDPA on head-shard
        # shapes: (B, hloc, N, D)
        attn = F.scaled_dot_product_attention(qh, kh, vh)  # (B, hloc, N, D)

        # merge heads back to (B, N, Hloc*D)
        attn = attn.permute(0,2,1,3).contiguous().view(B, -1, self.hloc * self.head_dim)

        # head->token shard back: (B, N_local, H)
        y_local = heads_to_tokens_alltoall(attn, world, H, self.hloc * self.head_dim)

        # output proj locally per token-shard
        y_local = self.o_proj(y_local)  # (B, N_local, H)
        return y_local

class SeqParMLP(nn.Module):
    def __init__(self, ref_mlp):
        super().__init__()
        self.gate_proj = nn.Linear(ref_mlp.gate_proj.in_features, ref_mlp.gate_proj.out_features, bias=False)
        self.up_proj   = nn.Linear(ref_mlp.up_proj.in_features,   ref_mlp.up_proj.out_features,   bias=False)
        self.down_proj = nn.Linear(ref_mlp.down_proj.in_features, ref_mlp.down_proj.out_features, bias=False)
    def forward(self, x):
        return self.down_proj(torch.silu(self.gate_proj(x)) * self.up_proj(x))

class SeqParBlock(nn.Module):
    def __init__(self, ref_block, config, rank, world):
        super().__init__()
        hs = ref_block.input_layernorm.weight.shape[0]
        self.rms1 = nn.RMSNorm(hs)
        self.rms2 = nn.RMSNorm(hs)
        self.attn = SeqParAttention(ref_block.self_attn, config, rank, world)
        self.mlp  = SeqParMLP(ref_block.mlp)
    def forward(self, x_local, position_ids_full):
        h = self.rms1(x_local)
        h = h + self.attn(h, position_ids_full)  # attn returns local tokens
        h2 = self.rms2(h)
        h = h + self.mlp(h2)
        return h

class SequenceParallelLlama(nn.Module):
    def __init__(self, ref_model, config, rank, world, num_layers=None):
        super().__init__()
        self.embed = nn.Embedding(ref_model.model.embed_tokens.num_embeddings,
                                  ref_model.model.embed_tokens.embedding_dim)
        layers = ref_model.model.layers if (num_layers is None) else ref_model.model.layers[:num_layers]
        self.layers = nn.ModuleList([SeqParBlock(lb, config, rank, world) for lb in layers])
        self.final_norm = nn.RMSNorm(ref_model.model.norm.weight.shape[0])
        self.lm_head = nn.Linear(ref_model.lm_head.in_features, ref_model.lm_head.out_features, bias=False)
        self.rank = rank; self.world = world

    def forward(self, input_ids_full):
        rank, world = self.rank, self.world
        B, S = input_ids_full.shape
        # full position ids (shared across ranks)
        position_ids = torch.arange(S, device=input_ids_full.device).unsqueeze(0).expand(B, S)

        # local token-shard
        input_ids_local, token_slice, S_local = split_sequence(input_ids_full, rank, world)
        x_local = self.embed(input_ids_local)  # (B, S_local, H)

        for layer in self.layers:
            x_local = layer(x_local, position_ids)  # attention internally does token<->head all_to_all

        # final norm + lm_head per local tokens
        x_local = self.final_norm(x_local)
        logits_local = self.lm_head(x_local)  # (B, S_local, V)

        # для сверки соберём полный логит (можно убрать на проде)
        parts = [torch.empty_like(logits_local) for _ in range(world)]
        dist.all_gather(parts, logits_local)
        logits_full = torch.cat(parts, dim=1)
        return logits_full

@torch.no_grad()
def copy_weights(ref_model, sp_model):
    sp_model.embed.weight.copy_(ref_model.model.embed_tokens.weight)
    sp_model.final_norm.load_state_dict(ref_model.model.norm.state_dict())
    sp_model.lm_head.weight.copy_(ref_model.lm_head.weight)
    for (rb, sb) in zip(ref_model.model.layers, sp_model.layers):
        sb.rms1.load_state_dict(rb.input_layernorm.state_dict())
        sb.rms2.load_state_dict(rb.post_attention_layernorm.state_dict())
        sb.attn.q_proj.weight.copy_(rb.self_attn.q_proj.weight)
        sb.attn.k_proj.weight.copy_(rb.self_attn.k_proj.weight)
        sb.attn.v_proj.weight.copy_(rb.self_attn.v_proj.weight)
        sb.attn.o_proj.weight.copy_(rb.self_attn.o_proj.weight)
        sb.mlp.gate_proj.weight.copy_(rb.mlp.gate_proj.weight)
        sb.mlp.up_proj.weight.copy_(rb.mlp.up_proj.weight)
        sb.mlp.down_proj.weight.copy_(rb.mlp.down_proj.weight)

def init_distributed(backend):
    if dist.is_available() and dist.is_initialized():
        return
    if all(k in os.environ for k in ["RANK","WORLD_SIZE","MASTER_ADDR","MASTER_PORT"]):
        # CUDA: set device if needed
        if backend == "nccl":
            local_rank = int(os.environ.get("LOCAL_RANK", 0))
            torch.cuda.set_device(local_rank)
        dist.init_process_group(backend=backend)
    else:
        dist.init_process_group(backend=backend, rank=0, world_size=1, init_method="file:///tmp/seqpar_init")

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--model', default='unsloth/Llama-3.2-1B')
    parser.add_argument('--backend', default='gloo')  # use nccl on multi-gpu
    parser.add_argument('--dtype', default='fp32', choices=['fp32','bf16','fp16'])
    parser.add_argument('--seq-len', type=int, default=256)
    parser.add_argument('--layers', type=int, default=2)
    parser.add_argument('--dummy', action='store_true')
    args = parser.parse_args()

    init_distributed(args.backend)
    rank, world = dist.get_rank(), dist.get_world_size()
    device = torch.device(f"cuda:{int(os.environ.get('LOCAL_RANK',0))}") if args.backend=="nccl" else torch.device("cpu")
    to_dtype = {'fp32': torch.float32, 'bf16': torch.bfloat16, 'fp16': torch.float16}[args.dtype]
    torch.manual_seed(42)

    if args.dummy:
        if rank == 0: print("Using dummy config")
        config = LlamaConfig(hidden_size=512, intermediate_size=4096, num_attention_heads=8, num_hidden_layers=4, vocab_size=32000)
        ref_model = LlamaForCausalLM(config).to(to_dtype).to(device)
        tokenizer = AutoTokenizer.from_pretrained('hf-internal-testing/llama-tokenizer')
        if tokenizer.pad_token_id is None: tokenizer.pad_token_id = tokenizer.eos_token_id
    else:
        ref_model = None; tokenizer = None
        for r in range(world):
            dist.barrier()
            if r != rank: continue
            ref_model = LlamaForCausalLM.from_pretrained(args.model, torch_dtype=to_dtype).to(device)
            tokenizer = AutoTokenizer.from_pretrained(args.model)
            if tokenizer.pad_token_id is None: tokenizer.pad_token_id = tokenizer.eos_token_id
            if rank == 0: print(f"Loaded {args.model}")
        dist.barrier()

    rope = LlamaRotaryEmbedding(ref_model.config).to(device)

    config = ref_model.config
    sp_model = SequenceParallelLlama(ref_model, config, rank, world, num_layers=(None if args.layers<=0 else args.layers)).to(device)
    copy_weights(ref_model, sp_model)

    # make seq_len divisible by world
    S = args.seq_len
    if S % world != 0:
        S = (S // world) * world
        if rank == 0: print(f"Adjusted seq-len to {S}")
    input_ids = torch.randint(0, ref_model.model.embed_tokens.num_embeddings, (1, S), device=device)

    with torch.no_grad():
        # reference (subset of layers) on full model for correctness
        x = ref_model.model.embed_tokens(input_ids)
        B, S, _ = x.shape
        position_ids = torch.arange(S, device=device).unsqueeze(0).expand(B, S)
        cos, sin = rope(x, position_ids)
        layers = ref_model.model.layers if len(sp_model.layers)==ref_model.config.num_hidden_layers else ref_model.model.layers[:len(sp_model.layers)]
        for lb in layers:
            # attention
            attn_out, *_ = lb.self_attn(lb.input_layernorm(x), position_embeddings=(cos, sin),  attention_mask=None)
            x = x + attn_out
            # mlp
            x = x + lb.mlp(lb.post_attention_layernorm(x))
        x = ref_model.model.norm(x)
        ref_logits = ref_model.lm_head(x)

        sp_logits = sp_model(input_ids)

    if rank == 0:
        diff = (sp_logits - ref_logits).abs().max().item()
        tol = 1e-4 if to_dtype==torch.float32 else 5e-3
        print(f"[SeqPar] max|logits diff| = {diff:.3e}")
        assert diff < tol, "Sequence-parallel forward mismatch"
        print("Sequence-parallel forward PASSED.")

if __name__ == "__main__":
    main()

Writing sequence_parallel_forward.py


In [3]:
#### Multi-GPU run examples / Примеры запуска

# CPU (2 ранга)
#!OMP_NUM_THREADS=1 torchrun --nproc_per_node 2 week05_large_models/sequence_parallel_forward.py --layers 2 --seq-len 256 --dtype fp32

# GPU (bf16)
#!torchrun --nproc_per_node 2 week05_large_models/sequence_parallel_forward.py --layers 2 --seq-len 512 --dtype bf16 --backend nccl

# Dummy tiny model (быстрый тест без скачивания весов)
!python sequence_parallel_forward.py --dummy --layers 2 --seq-len 128 --backend nccl



2025-09-22 20:29:18.040645: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1758572958.083221    1235 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1758572958.094185    1235 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1758572958.131272    1235 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1758572958.131327    1235 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1758572958.131338    1235 computation_placer.cc:177] computation placer alr

__Task 5 (1 point):__ Now use the script above to parallelize the entire training run. You are free to use other fine-tuning methods (e.g. LoRA or even full fine-tuning), as long as you can demonstrate that the loss goes down.

**Make sure you increase SEQUENCE_LENGTH as much as possible!** Even on a single GPU, you should be able to go into thousands, if not tens of thousands of tokens - and report the maximum sequence length with one and with multiple GPUs respectively.

If you don't have access to multiple GPUs, you may optionally submit a version that does training on a single GPU, but computes attention heads sequentially with gradient checkpointing - but if you do, please announce that you are using this option in bold, capital letters, so that the grader will notice it.

In [11]:
%%writefile sequence_parallel_train.py
import os, math, argparse
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributed as dist
from transformers import LlamaForCausalLM, AutoTokenizer, LlamaConfig
from transformers.models.llama.modeling_llama import LlamaRotaryEmbedding, apply_rotary_pos_emb

# ========== Distributed init ==========

def init_distributed(backend: str):
    if dist.is_available() and dist.is_initialized():
        return
    # set CUDA device if using NCCL
    if backend == "nccl":
        local_rank = int(os.environ.get("LOCAL_RANK", 0))
        torch.cuda.set_device(local_rank)
    if all(k in os.environ for k in ["RANK","WORLD_SIZE","MASTER_ADDR","MASTER_PORT"]):
        dist.init_process_group(backend=backend)
    else:
        # single-process fallback (useful in notebook)
        dist.init_process_group(backend=backend, init_method="file:///tmp/seqpar_train_init", rank=0, world_size=1)

# ========== Helpers ==========

def split_sequence(input_ids: torch.Tensor, rank: int, world: int):
    """Split (B,S) -> local (B,S_local)."""
    B, S = input_ids.shape
    assert S % world == 0, "Sequence length must be divisible by world_size"
    S_local = S // world
    sl = slice(rank * S_local, (rank + 1) * S_local)
    return input_ids[:, sl], sl, S_local

def tokens_to_heads_alltoall(t_local: torch.Tensor, world: int, hloc_vec: int):
    """
    Token-shard -> Head-shard.

    In:  t_local (B, N_local, H_total), where H_total = world * hloc_vec
    Out: th_local (B, N, hloc_vec) located on each rank (head shard for ALL tokens)
    """
    B, N_local, H_total = t_local.shape
    assert H_total % world == 0 and (H_total // world) == hloc_vec
    N = N_local * world

    # reshape to [B, N_local, world, hloc_vec] and unbind across 'world'
    t = t_local.view(B, N_local, world, hloc_vec)
    send_list = list(t.unbind(dim=2))  # world tensors, each (B, N_local, hloc_vec)

    recv_list = [torch.empty_like(send_list[0]) for _ in range(world)]
    dist.all_to_all(recv_list, send_list)  # permutation across ranks

    # concat along tokens -> (B, N, hloc_vec)
    th_local = torch.cat(recv_list, dim=1)
    return th_local

def heads_to_tokens_alltoall(th_local: torch.Tensor, world: int, hloc_vec: int):
    """
    Head-shard -> Token-shard.

    In:  th_local (B, N, hloc_vec) on each rank (local heads for ALL tokens)
    Out: t_local  (B, N_local, H_total) with H_total = world * hloc_vec
    """
    B, N, _ = th_local.shape
    assert N % world == 0
    N_local = N // world

    chunks = list(th_local.split(N_local, dim=1))          # world tensors of (B, N_local, hloc_vec)
    recv_list = [torch.empty_like(chunks[0]) for _ in range(world)]
    dist.all_to_all(recv_list, chunks)

    # concat along head shard dim -> (B, N_local, world * hloc_vec)
    t_local = torch.cat(recv_list, dim=2)
    return t_local

# ========== Sequence-parallel Attention with RoPE + 2x all_to_all ==========

class SeqParAttention(nn.Module):
    def __init__(self, ref_attn, config, rank: int, world: int):
        super().__init__()
        self.hidden_size = ref_attn.q_proj.in_features
        self.num_heads = ref_attn.num_heads
        assert self.hidden_size % self.num_heads == 0
        self.head_dim = self.hidden_size // self.num_heads
        assert self.num_heads % world == 0

        self.rank = rank
        self.world = world
        self.hloc = self.num_heads // world                  # local heads count
        self.hloc_vec = self.hloc * self.head_dim            # local heads hidden size

        # Replicated weights (teaching version; could be TP-sharded too)
        self.q_proj = nn.Linear(self.hidden_size, self.hidden_size, bias=False)
        self.k_proj = nn.Linear(self.hidden_size, self.hidden_size, bias=False)
        self.v_proj = nn.Linear(self.hidden_size, self.hidden_size, bias=False)
        self.o_proj = nn.Linear(self.hidden_size, self.hidden_size, bias=False)

        # RoPE
        self.rope = LlamaRotaryEmbedding(config)

    def forward(self, x_local: torch.Tensor, position_ids_full: torch.Tensor):
        """
        x_local:          (B, N_local, H)  -- token shard
        position_ids_full:(B, N)           -- full positions (same on all ranks)
        returns y_local:  (B, N_local, H)
        """
        B, N_local, H = x_local.shape
        world = self.world

        # 1) Local projections
        q = self.q_proj(x_local)   # (B, N_local, H)
        k = self.k_proj(x_local)
        v = self.v_proj(x_local)

        # 2) token->head shard via all_to_all (NO full S on each rank)
        q_h = tokens_to_heads_alltoall(q, world, self.hloc_vec)  # (B, N, hloc_vec)
        k_h = tokens_to_heads_alltoall(k, world, self.hloc_vec)
        v_h = tokens_to_heads_alltoall(v, world, self.hloc_vec)

        # 3) to (B, hloc, N, D)
        def to_heads(t):
            B_, N_, _ = t.shape
            return t.view(B_, N_, self.hloc, self.head_dim).permute(0, 2, 1, 3).contiguous()
        qh, kh, vh = to_heads(q_h), to_heads(k_h), to_heads(v_h)  # (B, hloc, N, D)

        # 4) RoPE on Q/K
        # rope API: get cos/sin by passing a sample shaped (B,N,dim), so use vh.transpose(1,2) as a placeholder
        cos, sin = self.rope(vh.transpose(1, 2), position_ids_full)  # (B,N,dim/2) internal shapes
        qh = apply_rotary_pos_emb(qh.transpose(1, 2), cos, sin, position_ids_full).transpose(1, 2)
        kh = apply_rotary_pos_emb(kh.transpose(1, 2), cos, sin, position_ids_full).transpose(1, 2)

        # 5) SDPA on local heads across ALL tokens
        attn = F.scaled_dot_product_attention(qh, kh, vh)  # (B, hloc, N, D)

        # 6) merge local heads -> (B, N, hloc_vec)
        attn = attn.permute(0, 2, 1, 3).contiguous().view(B, -1, self.hloc_vec)

        # 7) head->token shard back -> (B, N_local, H)
        y_local = heads_to_tokens_alltoall(attn, world, self.hloc_vec)

        # 8) output proj per token shard
        y_local = self.o_proj(y_local)
        return y_local

# ========== MLP (replicated) ==========

class SeqParMLP(nn.Module):
    def __init__(self, ref_mlp):
        super().__init__()
        self.gate_proj = nn.Linear(ref_mlp.gate_proj.in_features, ref_mlp.gate_proj.out_features, bias=False)
        self.up_proj   = nn.Linear(ref_mlp.up_proj.in_features,   ref_mlp.up_proj.out_features,   bias=False)
        self.down_proj = nn.Linear(ref_mlp.down_proj.in_features, ref_mlp.down_proj.out_features, bias=False)
    def forward(self, x):
        return self.down_proj(torch.silu(self.gate_proj(x)) * self.up_proj(x))

# ========== Block (sequence-local residual path) ==========

class SeqParBlock(nn.Module):
    def __init__(self, ref_block, config, rank, world):
        super().__init__()
        hs = ref_block.input_layernorm.normalized_shape[0]
        self.rms1 = nn.RMSNorm(hs)
        self.rms2 = nn.RMSNorm(hs)
        self.attn = SeqParAttention(ref_block.self_attn, config, rank, world)
        self.mlp  = SeqParMLP(ref_block.mlp)
    def forward(self, x_local, position_ids_full):
        h = self.rms1(x_local)
        h = x_local + self.attn(h, position_ids_full)   # attention returns local tokens
        h2 = self.rms2(h)
        h = h + self.mlp(h2)
        return h

# ========== Model wrapper for training (returns LOCAL logits) ==========

class SequenceParallelTrainModel(nn.Module):
    """
    Возвращает ЛОКАЛЬНЫЕ logits (B, T_local, V).
    Снаружи считаем loss по локальным меткам и делаем all_reduce(AVG).
    """
    def __init__(self, ref_model, rank, world, num_layers=None):
        super().__init__()
        self.rank = rank
        self.world = world
        self.config = ref_model.config
        self.embed = nn.Embedding(ref_model.model.embed_tokens.num_embeddings,
                                  ref_model.model.embed_tokens.embedding_dim)
        layers = ref_model.model.layers if (num_layers is None) else ref_model.model.layers[:num_layers]
        self.layers = nn.ModuleList([SeqParBlock(lb, ref_model.config, rank, world) for lb in layers])
        self.final_norm = nn.RMSNorm(ref_model.model.norm.normalized_shape[0])
        self.lm_head = nn.Linear(ref_model.lm_head.in_features, ref_model.lm_head.out_features, bias=False)

    def forward(self, input_ids_full: torch.Tensor, prompt_emb_full: torch.Tensor | None = None):
        """
        input_ids_full: (B, S_real)
        prompt_emb_full (optional): (B, V, H) — виртуальные токены, уже собранные глобально
        returns local logits: (B, T_local, Vocab), где T = (V + S_real)
        """
        rank, world = self.rank, self.world
        device = input_ids_full.device
        B, S_real = input_ids_full.shape

        # embed real tokens locally
        input_ids_local, real_slice_local, S_local = split_sequence(input_ids_full, rank, world)
        x_local = self.embed(input_ids_local)  # (B, S_local, H)

        # если есть prompt в начале, нужно добавить его "впереди" всей последовательности
        if prompt_emb_full is not None:
            V = prompt_emb_full.size(1)
            T = V + S_real
            assert T % world == 0, "For simplicity, (V+S) must divide world_size"
            T_local = T // world
            # построим локальный срез глобальной последовательности (prompt + tokens)
            full_local_slice = slice(rank * T_local, (rank + 1) * T_local)

            # Чтобы получить локальный кусок после добавления prompt, сначала соберём "реальные" токены
            # в глобальный (учебный способ), потом склеим с prompt и отрежем локальный фрагмент.
            # Это одноразовый gather с небольшим оверхедом, основная экономия идёт из 2x all_to_all в attention.
            parts = [torch.empty_like(x_local) for _ in range(world)]
            dist.all_gather(parts, x_local)                  # (B, S_local, H) x world
            x_full = torch.cat(parts, dim=1)                 # (B, S_real, H)
            x_full = torch.cat([prompt_emb_full, x_full], dim=1)  # (B, V+S_real, H)
            x_local = x_full[:, full_local_slice, :]         # (B, T_local, H)

            # positions for RoPE for the full (prompt+tokens)
            pos_ids_full = torch.arange(T, device=device).unsqueeze(0).expand(B, T)
        else:
            # без prompt — просто локальный шард
            T = S_real
            T_local = S_local
            pos_ids_full = torch.arange(T, device=device).unsqueeze(0).expand(B, T)
            # локальный срез относительно full T
            full_local_slice = slice(rank * T_local, (rank + 1) * T_local)

        # pass through blocks (attention uses 2x all_to_all internally, takes full position ids)
        for layer in self.layers:
            x_local = layer(x_local, pos_ids_full)

        # head on local tokens, then return local logits
        x_local = self.final_norm(x_local)
        logits_local = self.lm_head(x_local)  # (B, T_local, Vocab)
        return logits_local, full_local_slice, T, T_local

# ========== Weight copy (replicated) ==========

@torch.no_grad()
def copy_weights(ref_model: LlamaForCausalLM, sp_model: SequenceParallelTrainModel):
    sp_model.embed.weight.copy_(ref_model.model.embed_tokens.weight)
    sp_model.final_norm.load_state_dict(ref_model.model.norm.state_dict())
    sp_model.lm_head.weight.copy_(ref_model.lm_head.weight)
    for rb, sb in zip(ref_model.model.layers, sp_model.layers):
        sb.rms1.load_state_dict(rb.input_layernorm.state_dict())
        sb.rms2.load_state_dict(rb.post_attention_layernorm.state_dict())
        sb.attn.q_proj.weight.copy_(rb.self_attn.q_proj.weight)
        sb.attn.k_proj.weight.copy_(rb.self_attn.k_proj.weight)
        sb.attn.v_proj.weight.copy_(rb.self_attn.v_proj.weight)
        sb.attn.o_proj.weight.copy_(rb.self_attn.o_proj.weight)
        sb.mlp.gate_proj.weight.copy_(rb.mlp.gate_proj.weight)
        sb.mlp.up_proj.weight.copy_(rb.mlp.up_proj.weight)
        sb.mlp.down_proj.weight.copy_(rb.mlp.down_proj.weight)

# ========== Utilities ==========

@torch.no_grad()
def freeze_except(module: nn.Module, predicate):
    for name, p in module.named_parameters():
        p.requires_grad_(predicate(name))

def build_pair_from_text(tokenizer, path: str, seq_len: int, device):
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    text = open(path, 'r', encoding='utf-8', errors='ignore').read()
    ids = tokenizer(text, return_tensors='pt')['input_ids'][0]
    if ids.numel() < seq_len + 1:
        # loop the text if too short (simple trick)
        reps = (seq_len + 1 + ids.numel() - 1) // ids.numel()
        ids = ids.repeat(reps)
    ids = ids[:seq_len + 1]
    input_ids = ids[:-1].unsqueeze(0).to(device)   # (B=1, S)
    labels    = ids[1:].unsqueeze(0).to(device)    # (B=1, S)
    return input_ids, labels

# ========== Prompt Tuning parameters ==========

class PromptTuner(nn.Module):
    """Простая обучаемая матрица виртуальных токенов (B-shared)."""
    def __init__(self, virtual_tokens: int, hidden_size: int):
        super().__init__()
        self.emb = nn.Embedding(virtual_tokens, hidden_size)
        nn.init.normal_(self.emb.weight, std=0.02)
        self.virtual_tokens = virtual_tokens
    def forward(self, batch_size: int, device: torch.device):
        ids = torch.arange(self.virtual_tokens, device=device).view(1, -1).expand(batch_size, -1)
        return self.emb(ids)  # (B, V, H)

# ========== Argparse ==========

def parse_args():
    ap = argparse.ArgumentParser()
    ap.add_argument('--model', default='unsloth/Llama-3.2-1B')
    ap.add_argument('--backend', default='gloo')   # use 'nccl' for multi-gpu
    ap.add_argument('--dtype', default='fp32', choices=['fp32', 'bf16', 'fp16'])
    ap.add_argument('--seq-len', type=int, default=1024)
    ap.add_argument('--layers', type=int, default=4)
    ap.add_argument('--virtual-tokens', type=int, default=32)
    ap.add_argument('--steps', type=int, default=20)
    ap.add_argument('--lr', type=float, default=5e-4)
    ap.add_argument('--text-path', default='ulysses.txt')
    ap.add_argument('--dummy', action='store_true')
    ap.add_argument('--grad-ckpt', action='store_true', help='(optional) enable gradient checkpointing per block')
    return ap.parse_args()

def str_to_dtype(s): return {'fp32': torch.float32, 'bf16': torch.bfloat16, 'fp16': torch.float16}[s]

# ========== Main training loop ==========

def main():
    args = parse_args()
    init_distributed(args.backend)
    rank = dist.get_rank(); world = dist.get_world_size()

    use_cuda = (args.backend == "nccl") and torch.cuda.is_available()
    device = torch.device(f"cuda:{int(os.environ.get('LOCAL_RANK',0))}") if use_cuda else torch.device("cpu")
    torch.manual_seed(123 + rank)
    dtype = str_to_dtype(args.dtype)

    # Load model/tokenizer sequentially to lower peak RAM
    if args.dummy:
        if rank == 0: print("Using dummy config")
        cfg = LlamaConfig(hidden_size=512, intermediate_size=1536, num_attention_heads=8,
                          num_hidden_layers=8, vocab_size=32000)
        ref_model = LlamaForCausalLM(cfg).to(dtype).to(device)
        tokenizer = AutoTokenizer.from_pretrained('hf-internal-testing/llama-tokenizer')
        if tokenizer.pad_token_id is None: tokenizer.pad_token_id = tokenizer.eos_token_id
    else:
        ref_model = None; tokenizer = None
        for r in range(world):
            dist.barrier()
            if r != rank: continue
            ref_model = LlamaForCausalLM.from_pretrained(args.model, torch_dtype=dtype).to(device)
            tokenizer = AutoTokenizer.from_pretrained(args.model)
            if tokenizer.pad_token_id is None: tokenizer.pad_token_id = tokenizer.eos_token_id
            if rank == 0: print("Model loaded")
        dist.barrier()

    # Build SP wrapper
    sp = SequenceParallelTrainModel(ref_model, rank, world, num_layers=(None if args.layers < 0 else args.layers)).to(dtype).to(device)
    copy_weights(ref_model, sp)

    # Freeze everything in SP model (prompt tuning only)
    for p in sp.parameters(): p.requires_grad_(False)

    # Prompt tuner (replicated on all ranks)
    prompt = PromptTuner(args.virtual_tokens, sp.config.hidden_size).to(dtype).to(device)

    # AMP
    use_autocast = (dtype in (torch.float16, torch.bfloat16))
    scaler = torch.cuda.amp.GradScaler(enabled=(dtype == torch.float16))

    # Optimizer only for prompt
    optim = torch.optim.AdamW(prompt.parameters(), lr=args.lr)

    # Data
    if rank == 0 and not os.path.exists(args.text_path):
        os.system(f"wget -q https://www.gutenberg.org/cache/epub/4300/pg4300.txt -O {args.text_path}")
    dist.barrier()
    input_ids_full, labels_full = build_pair_from_text(tokenizer, args.text_path, args.seq_len, device)

    # Make (V + S) divisible by world
    total_seq = args.virtual_tokens + input_ids_full.size(1)
    if total_seq % world != 0:
        new_S = (total_seq // world) * world - args.virtual_tokens
        if new_S <= 0:
            raise ValueError("Cannot satisfy divisibility; decrease virtual tokens or world size")
        if rank == 0:
            print(f"[Adjust] seq_len {input_ids_full.size(1)} -> {new_S} to satisfy (V+S) % world == 0")
        input_ids_full = input_ids_full[:, :new_S]
        labels_full    = labels_full[:, :new_S]

    # Optional grad checkpointing (simple per-layer wrapper)
    if args.grad_ckpt:
        def ckpt_layer(layer):
            def wrapper(x, pos):
                return torch.utils.checkpoint.checkpoint(lambda _x: layer(_x, pos), x, use_reentrant=False)
            return wrapper
        for i, blk in enumerate(sp.layers):
            sp.layers[i].forward = ckpt_layer(blk)

    sp.train(); prompt.train()

    loss_fn = nn.CrossEntropyLoss(reduction="mean")

    for step in range(1, args.steps + 1):
        optim.zero_grad(set_to_none=True)

        with torch.autocast(device_type=("cuda" if use_cuda else "cpu"),
                            dtype=(torch.bfloat16 if dtype==torch.bfloat16 else torch.float16),
                            enabled=use_autocast):
            # Build prompt embeddings (B,V,H), same on all ranks
            B = input_ids_full.size(0)
            prompt_emb_full = prompt(B, device)  # (B, V, H)

            # Forward returns LOCAL logits and slice mapping into global time axis
            logits_local, local_slice, T, T_local = sp(input_ids_full, prompt_emb_full=prompt_emb_full)
            # Labels are only for REAL tokens; prompt occupies the first V positions.
            V = args.virtual_tokens
            # We want labels aligned to positions [V .. V+S-1] in the global T timeline.
            # Intersect our local slice with [V .. V+S-1] and compute loss for that window.
            start = max(local_slice.start, V)
            end   = min(local_slice.stop,  V + labels_full.size(1))
            if start < end:
                # slice into local logits
                # local index = (global index - local_slice.start)
                rel_start = start - local_slice.start
                rel_end   = end   - local_slice.start
                logits_part = logits_local[:, rel_start:rel_end, :]                         # (B, Lp, Vocab)
                labels_part = labels_full[:, (start - V):(end - V)]                         # (B, Lp)
                loss = loss_fn(logits_part.reshape(-1, logits_part.size(-1)),
                               labels_part.reshape(-1))
            else:
                # this rank has no supervised tokens (only prompt zone) — 0 loss
                loss = logits_local.sum() * 0.0

        # average loss across ranks for logging
        loss_for_log = loss.detach()
        dist.all_reduce(loss_for_log, op=dist.ReduceOp.AVG)

        # backward + grad sync (prompt params only)
        if dtype == torch.float16:
            scaler.scale(loss).backward()
        else:
            loss.backward()

        # manually all_reduce grads for prompt so replicas stay in sync
        for p in prompt.parameters():
            if p.grad is not None:
                dist.all_reduce(p.grad, op=dist.ReduceOp.AVG)

        # step
        if dtype == torch.float16:
            scaler.step(optim)
            scaler.update()
        else:
            optim.step()

        if rank == 0:
            print(f"step={step:03d}  loss={loss_for_log.item():.4f}")

    if rank == 0:
        print("Training demo complete. Loss should go down at least a bit 🙂")

if __name__ == "__main__":
    main()

Writing sequence_parallel_train.py


In [12]:
!torchrun --nproc_per_node=2 sequence_parallel_train.py --backend gloo --seq-len 512 --layers 2 --dummy --steps 5

W0922 13:46:07.770000 12156 torch/distributed/run.py:774] 
W0922 13:46:07.770000 12156 torch/distributed/run.py:774] *****************************************
W0922 13:46:07.770000 12156 torch/distributed/run.py:774] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0922 13:46:07.770000 12156 torch/distributed/run.py:774] *****************************************
2025-09-22 13:46:16.392260: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1758548776.462228   12163 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1758548776.490099   12163 cuda_blas.cc:1407] Unable to register cuBLAS facto

```

```

```

```

```

```


### Optional: bonus tasks

There are many routes to further improve the training/inference code. You may (but you don't have to) implement any combination of them for bonus points.

However, please not that the total points for this week's entire assignment (part 1 & 2) are **capped at 14**.

__Bonus task: parallel key-value caching (1 point).__ In tasks 2 and 3, you implement tensor parallelism for attention forward pass and perform inference with re-computation. However, real world inference engines use [KV caching](https://huggingface.co/docs/transformers/main/en/kv_cache) - keeping key and value caches from past tokens and only processing the new token each time.

For this task, you will have to implement this type of parallelism for either torch.distributed or DTensor implementation of attention $-$ simply cache the heads already assigned to each rank. To get the grade, you will need to demonstrate that the model generates a sensible text with any cache (via past_key_values=).

__Bonus task: pipeline parallelism (1-2 points):__ In tasks 1-3, you've implemented symmetric model parallelism, aka Tensor Parallelism. However, there is another way to partition model parameters $-$ assign entire layers to each rank and run them in a pipeline. This can be faster, especially if you are running

For 1 point, check out [torch.distributed.pipelinging](https://pytorch.org/docs/stable/distributed.pipelining.html), [DeepSpeed pipelining](https://deepspeed.readthedocs.io/en/latest/pipeline.html) or [torchgpipe](https://github.com/kakaobrain/torchgpipe) and demonstrate that you can run or fine-tune a model that would not fit into a single GPU (you will need multuple devices for this!).

For 2 points, compare different pipelining schedules in terms of training throughput: use GPipe as a baseline and try ScheduleInterleaved1F1B (or a more advanced pipeline of your choosing).

__Bonus task: better sequence parallelism (2 points).__ In tasks 4 and 5, you implemented basic sequence parallelism. However, there are multiple ways you can improve that technique for further memory savings or better device utilization.

For 1 point, implement combined tensor + sequence parallelism and compare results with naive sequence parallelism.

For 2 points, implement [Ring Attention](https://arxiv.org/abs/2310.01889) *or* integrate computation-communication overlap from [FLUX](https://arxiv.org/abs/2406.06858) and measure the speed and memory trade-offs.

In [2]:
# Demo: compare generation with & without KV cache (dummy model by default)
!python tensor_parallel_llama.py --dummy --prompt "KV cache demo" --max-new-tokens 8 --time-compare --use-cache

/opt/anaconda3/envs/dl_end/lib/python3.10/site-packages/torch/cuda/__init__.py:56: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/opt/anaconda3/envs/dl_end/lib/python3.10/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
Traceback (most recent call last):
  File "/Users/atom/dev/efficient-dl-systems/week05_large_models/tensor_parallel_llama.py", line 378, in <module>
    main()
  File "/Users/atom/dev/efficient-dl-systems/week05_large_models/tensor_parallel_llama.py", line 323, in main
    init_distributed(args.backend)
  File "/Users/atom/dev/efficient-dl-systems/week05_large_models/tensor_parallel_llama.py", line 264, in init_distributed
    dist.init_process_gro

### Bonus: Tensor Parallel KV Cache

Ниже демонстрируется расширение Task 2: добавлена поддержка `past_key_values` в файл `tensor_parallel_llama.py`.
Каждый rank кэширует только свои головы (K,V) — дополнительной синхронизации не требуется, так как головы распределены дизъюнктно.

Проверяем:
1. Генерация без кэша (recompute каждого шага)
2. Генерация с кэшем
3. Сравнение времени и идентичность последовательностей (токены должны совпадать)

Для быстрой проверки используем `--dummy` (маленькая модель); для настоящей — указываем `--model-name`.

Пример (2 GPU):
```bash
torchrun --nproc_per_node 2 tensor_parallel_llama.py \
  --dummy --prompt "Once upon a time" \
  --max-new-tokens 32 --time-compare --use-cache
```
Ожидаемо: версия с кэшем быстрее (меньше задержка на токен), а вывод совпадает.


In [ ]:
# Demo: LoRA + prompt tuning (dummy)
!python sequence_parallel_train_lora.py --dummy --layers 2 --seq-len 256 --steps 5 --lora --lora-r 4 --virtual-tokens 8 --lora-targets attn.q_proj

### Bonus: LoRA + Sequence Parallel Training

Добавлен файл `sequence_parallel_train_lora.py`:
- Поддержка Prompt Tuning (`--virtual-tokens N`) и/или LoRA (`--lora`).
- Параметры LoRA: `--lora-r`, `--lora-alpha`, `--lora-dropout`, `--lora-targets` (подстроки в пути модуля, например `attn.q_proj`).
- Выводится число и доля обучаемых параметров.

Примеры:
```bash
# Быстрый dummy тест (CPU/GPU)
python sequence_parallel_train_lora.py --dummy --layers 2 --seq-len 512 --steps 10 \
  --lora --lora-r 8 --lora-targets attn.q_proj,attn.v_proj --virtual-tokens 16

# 2 GPU bf16
torchrun --nproc_per_node 2 sequence_parallel_train_lora.py --dummy --layers 4 --seq-len 2048 \
  --steps 30 --backend nccl --dtype bf16 --lora --lora-r 16 \
  --lora-targets attn.q_proj,mlp.gate_proj --virtual-tokens 32
```
Ожидаемо: loss убывает, trainable params << total.
